# Test of the google/gemma-3-4b-pt model

In [2]:
# pip install accelerate

from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from PIL import Image
import requests
import torch

model_id = "google/gemma-3-4b-pt"

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
image = Image.open(requests.get(url, stream=True).raw)


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
model = Gemma3ForConditionalGeneration.from_pretrained(model_id,
                                                       torch_dtype=torch.float16,
                                                       device_map="auto" ).eval()
processor = AutoProcessor.from_pretrained(model_id)

prompt = "<start_of_image> in this image, there is"
model_inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

input_len = model_inputs["input_ids"].shape[-1]

Loading weights: 100%|██████████| 883/883 [00:06<00:00, 142.94it/s, Materializing param=model.vision_tower.vision_model.post_layernorm.weight]                      
Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [9]:
with torch.inference_mode():
    generation = model.generate(**model_inputs, max_new_tokens=100, do_sample=False)
    # generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

['\n\n\n\n in this image, there is']


In [10]:
print(decoded)

['\n\n\n\n in this image, there is']
